<a href="https://colab.research.google.com/github/hojuelita/text_classification/blob/main/Ejemplo_vectorizaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Métodos de embedding

In [21]:

!pip install gensim
# recuperamos el texto desde el git
import requests
import pandas as pd
import re # Added import for regular expressions

import gensim
from gensim.models import Word2Vec
import nltk


url_texto = "https://raw.githubusercontent.com/hojuelita/text_classification/refs/heads/main/data/Cuentos/Bajo_las_luces_del_sol_naciente.txt"
response = requests.get(url_texto)
texto = response.text

print("Original text:")

def clean_text(text, minusculas=True):
    # Remove everything inside brackets [...]
    text = re.sub(r"\[.*\]", '', text)
    # Remove everything inside parenthesis (...)
    text = re.sub(r'\(.*\)', '', text)
    # Remove continuous asterisks *
    text = re.sub(r'\*+', '', text)
    # Optional: remove leading/trailing whitespace that might result from cleaning
    if(minusculas):
      text = text.lower()
    text = text.strip()
    return text

def split_paragraphs(text):
    """
    Split the text into paragraphs based on double newlines.
    """
    paragraphs = text.split('\n\n')
    splitted = {}
    for j,par in enumerate(paragraphs):
      splitted[j] = re.sub("\n", "", par.strip())
    return splitted

cleaned_texto = clean_text(texto, minusculas=True)
print("\nCleaned text:")
print(cleaned_texto)

Original text:

Cleaned text:
bajo las luces del sol naciente



      los cerezos florecían, y entre sus ramas
    alegres se divisaba un monte azul.


era el país de oro y seda, y en el aire fino como de cristal volaban las
cigüeñas, y se esponjaban los crisantemos del biombo. los cerezos
florecían, y entre sus ramas alegres se divisaba un monte azul. una rana
de madera labrada era igual a las ranas del pantano. sobre la laca negra
corría un arroyo dorado. muñecas de carne, con la cabellera atravesada
por alfileres áureos, hacían reverencias sonrientes, y gestos menudos.
en las casas de papel, en la ignorancia feliz del pudor, se bañaban las
niñas. cortesanas ingenuas servían el té en tacitas de liliput. en los
«kimonos» historiados se envolvían cuerpos casi impúberes e
inocentemente venales. se hablaba de un viejo llamado hokusai, que se
llamaba a sí mismo «el loco del dibujo». floreros raros se llenaban de
flores extrañas ante los budhas risueños. nobles daimios hacían lucir al
sol

In [33]:
# vamos a limpiar el texto quitando caraceres especiales,
# todo lo que esté entre []
parrafos = split_paragraphs(cleaned_texto)
parrafos


{0: 'bajo las luces del sol naciente',
 1: '',
 2: 'los cerezos florecían, y entre sus ramas    alegres se divisaba un monte azul.',
 3: 'era el país de oro y seda, y en el aire fino como de cristal volaban lascigüeñas, y se esponjaban los crisantemos del biombo. los cerezosflorecían, y entre sus ramas alegres se divisaba un monte azul. una ranade madera labrada era igual a las ranas del pantano. sobre la laca negracorría un arroyo dorado. muñecas de carne, con la cabellera atravesadapor alfileres áureos, hacían reverencias sonrientes, y gestos menudos.en las casas de papel, en la ignorancia feliz del pudor, se bañaban lasniñas. cortesanas ingenuas servían el té en tacitas de liliput. en los«kimonos» historiados se envolvían cuerpos casi impúberes einocentemente venales. se hablaba de un viejo llamado hokusai, que sellamaba a sí mismo «el loco del dibujo». floreros raros se llenaban deflores extrañas ante los budhas risueños. nobles daimios hacían lucir alsol curvos sables de largo puñ

Una vez que tengamos los párrafos ya separados podemos vectorizarlos, lo vamos a hacer primero con word2vec pero también vamos a probarlo con FastText y Roberta

## Word2Vec

In [8]:
!pip install gensim

In [41]:
# modelo para oraciones de nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# revisar el modelo usado por spacy

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [52]:
nltk.sent_tokenize(str(parrafos[0]))[0].split()

['bajo', 'las', 'luces', 'del', 'sol', 'naciente']

In [53]:
# tokenizacion en palabras
corpus = []
corpus_oraciones = []
for parrafo in parrafos.values():
  print(parrafo)
  try:
    corpus_oraciones.append( nltk.sent_tokenize(parrafo)[0].split() )
  except:
    print(parrafo)
  oraciones = [palabra for oracion in nltk.sent_tokenize(parrafo) for palabra in oracion.split()]
  corpus += oraciones
  print(corpus)

bajo las luces del sol naciente
['bajo', 'las', 'luces', 'del', 'sol', 'naciente']


['bajo', 'las', 'luces', 'del', 'sol', 'naciente']
los cerezos florecían, y entre sus ramas    alegres se divisaba un monte azul.
['bajo', 'las', 'luces', 'del', 'sol', 'naciente', 'los', 'cerezos', 'florecían,', 'y', 'entre', 'sus', 'ramas', 'alegres', 'se', 'divisaba', 'un', 'monte', 'azul.']
era el país de oro y seda, y en el aire fino como de cristal volaban lascigüeñas, y se esponjaban los crisantemos del biombo. los cerezosflorecían, y entre sus ramas alegres se divisaba un monte azul. una ranade madera labrada era igual a las ranas del pantano. sobre la laca negracorría un arroyo dorado. muñecas de carne, con la cabellera atravesadapor alfileres áureos, hacían reverencias sonrientes, y gestos menudos.en las casas de papel, en la ignorancia feliz del pudor, se bañaban lasniñas. cortesanas ingenuas servían el té en tacitas de liliput. en los«kimonos» historiados se envolvían cuerpos casi impúberes

In [54]:
len(corpus_oraciones)

12

In [55]:
# instanciamos el modelo
# usando las oraciones
modelo = Word2Vec(sentences=corpus_oraciones, vector_size=100, window=5, min_count=1, workers=4)


In [56]:
modelo.wv.key_to_index

{'y': 0,
 'la': 1,
 'de': 2,
 'a': 3,
 'se': 4,
 'los': 5,
 'el': 6,
 'las': 7,
 'con': 8,
 'en': 9,
 'del': 10,
 'tenemos': 11,
 'que': 12,
 'un': 13,
 'sangre': 14,
 'civilización': 15,
 'como': 16,
 'aire': 17,
 'triunfado...': 18,
 'ha': 19,
 'laguerra...': 20,
 'amor': 21,
 'desafío,': 22,
 'unidos': 23,
 'estados': 24,
 'mira': 25,
 'sí!': 26,
 '¡oh,': 27,
 'matamos!': 28,
 'nosotros': 29,
 'nuestrosmuertos,': 30,
 'enrojecieron': 31,
 'china': 32,
 'mares': 33,
 'cañones.¡los': 34,
 'torpedos,': 35,
 'acorazados,': 36,
 'hoy': 37,
 'periodistaparisiense,': 38,
 'hayashi': 39,
 'japonés': 40,
 'afiladamente': 41,
 'sonriendo': 42,
 'dice': 43,
 '«hoy,': 44,
 'nuevo.': 45,
 'japón': 46,
 'convirtió': 47,
 'viejo,': 48,
 'eljapón': 49,
 'occidente': 50,
 'vida': 51,
 'despertó': 52,
 'cañonazos': 53,
 'virtudes.': 54,
 'antiguas': 55,
 'nuevos': 56,
 'corazones': 57,
 'enlos': 58,
 'día': 59,
 'cada': 60,
 'así': 61,
 'infundiendo': 62,
 'comprendían,': 63,
 'generaciones': 64,
 'c

In [62]:
modelo.wv.get_vector('naciente')

array([-2.4923244e-03, -8.4821889e-03,  4.4336339e-04, -3.9680949e-03,
       -3.4565197e-03, -2.8017221e-03,  3.5536550e-03, -5.1548784e-03,
        2.3390593e-03, -3.6041799e-04,  4.4985772e-03,  6.5361685e-04,
       -9.7064031e-03,  2.5318908e-03,  5.1624761e-03,  6.5916264e-03,
       -8.0140717e-03, -3.8585570e-03,  1.4429373e-03,  1.1331057e-03,
        6.5297773e-03, -6.9996291e-03, -8.5125808e-03,  1.6597336e-03,
        9.2322333e-04, -7.4461550e-03,  1.1465665e-03,  3.9811367e-03,
       -1.1205528e-03, -5.8863708e-03,  9.7228410e-03, -1.1608504e-03,
       -7.9003151e-04,  8.1289094e-03, -8.2367165e-03, -6.0728993e-03,
        4.6309801e-03, -7.4849287e-03, -8.9399237e-03,  4.6473769e-03,
       -3.9397795e-03,  1.8107441e-03, -3.2674541e-06,  7.5407946e-03,
       -1.4738687e-03, -6.9683148e-03,  2.9310163e-03,  9.2326012e-03,
       -3.1681976e-03,  2.6466798e-03, -9.1934893e-03,  8.8462718e-03,
        7.9066688e-03, -6.2342132e-03, -8.2470812e-03,  7.2505670e-03,
      

In [63]:
modelo.wv.most_similar('naciente')

[('intelectual', 0.32565268874168396),
 ('corazones', 0.24815185368061066),
 ('mundo', 0.2217925488948822),
 ('dice', 0.22084926068782806),
 ('país', 0.17780400812625885),
 ('sus', 0.15385045111179352),
 ('las', 0.14725659787654877),
 ('triunfado...', 0.1421714723110199),
 ('así', 0.136926531791687),
 ('japonés', 0.13322386145591736)]

# Tarea

Tarea en 3 etapas
1. Poner nb en repo para poder cargarlo desde colab
1. Cargar cada uno de los archivos de texto (novelas y cuentos)
    1.1 Hacer limpieza de texto para quitar ya sea los asteriscos * * * o las anotaciones entre brackets [] y lo que consideres
2. Separar los textos en oraciones ajenas en un objeto en memoria (salvar las oraciones en disco)
3. Entrenar w2v con este dataset. Hacer pruebas de similaridad.
    3.1 Si se ven bien salvar los vectores con FAISS